# 04 アルゴリズムと計算量（教科書の探索・ソート）

## 1. 研究目的（RQ4）

教科書の**探索・ソート**アルゴリズムを実行し、**理論計算量**（O 記法）と**実測時間**の関係を調べます。

- 線形探索 $O(n)$ と二分探索 $O(\log n)$
- バブルソート・挿入ソート $O(n^2)$ と、マージソート・`sorted()`・`np.sort()` $O(n \log n)$
- 入力の並び（ランダム・ソート済み・逆順・重複が多い）による最良・平均・最悪の場合

> 探索・ソートは、前の比較の結果で次の処理が決まる**逐次的な**アルゴリズムです。行列積のように GPU・TPU で単純に並列化できる計算ではないので、この Notebook では CPU だけを使います。

## 2. 理論的背景

| アルゴリズム | 教科書のファイル | 最良 | 平均 | 最悪 | 備考 |
| --- | --- | --- | --- | --- | --- |
| 線形探索 `seq_search` | `chap03/ssearch_while.py` | $O(1)$ | $O(n)$ | $O(n)$ | ソート不要 |
| 二分探索 `bin_search` | `chap03/bsearch.py` | $O(1)$ | $O(\log n)$ | $O(\log n)$ | **ソート済みが前提** |
| バブルソート `bubble_sort` | `chap06/bubble_sort1.py` | $O(n^2)$ | $O(n^2)$ | $O(n^2)$ | この版は途中で打ち切らない |
| 挿入ソート `insertion_sort` | `chap06/insertion_sort.py` | $O(n)$ | $O(n^2)$ | $O(n^2)$ | ソート済みに近いと速い |
| マージソート `merge_sort` | `chap06/merge_sort.py` | $O(n \log n)$ | $O(n \log n)$ | $O(n \log n)$ | 作業用配列が必要 |
| `sorted()`（Timsort） | Python 組み込み | $O(n)$ | $O(n \log n)$ | $O(n \log n)$ | C 言語で実装 |
| `np.sort()` | NumPy | | $O(n \log n)$ | | C 言語で実装、配列が対象 |

### $O(n)$ と $O(\log n)$

$n$ が 1,000 倍になると、$O(n)$ の時間はほぼ 1,000 倍、$O(\log n)$ の時間は $\log_2$ が約 10 増えるだけです（$\log_2 1000 \approx 10$、$\log_2 10^6 \approx 20$）。

### アルゴリズムの改善と実装の高速化

- **アルゴリズムの改善**（$O(n^2)$ → $O(n \log n)$）は、$n$ が大きいほど効果が大きくなる。
- **実装の高速化**（Python → C 言語・NumPy）は、定数倍を小さくする。$n$ が十分大きいと、遅い言語の良いアルゴリズムが、速い言語の悪いアルゴリズムに勝つ。

## 3. 実験環境（最初に必ず実行）

このセルは、E2 リポジトリの `.py`（`cpu_basics.py`、`numpy_benchmark.py` など）を import できるように、
**作業ディレクトリ（E2 のルート）を自動で探して設定**します。

| 実行場所 | 準備 |
| --- | --- |
| ローカルの Jupyter | `notebooks/` の中で開けば、そのまま動きます |
| Colab（方法A：GitHub から clone） | 下のセルの `REPO_URL` に**自分たちのチームのリポジトリの URL** を入れる（`REPO_SUBDIR` に E2 のフォルダ名） |
| Colab（方法B：ZIP をアップロード） | E2 フォルダを ZIP にして `/content` にアップロードしておく（またはこのセルの実行中にアップロード） |

> ⚠️ `textbook/`（教科書のコード）は**公開リポジトリには含めない**約束です。
> `textbook/` がないと `algorithm_benchmark.py` の import が失敗します（Notebook 04・05 の一部）。
> その場合は方法B（ZIP）を使うか、授業で配布された `textbook/` を同じ場所に置いてください。

In [ ]:
import os
import sys
import glob
import zipfile
import subprocess
from pathlib import Path

# ---- 方法A：GitHub から clone する場合だけ設定する（URL は自分たちのものを入れる）----
REPO_URL = ""                              # 例："https://github.com/<team>/<repo>.git"（空なら clone しない）
REPO_SUBDIR = "e2-computing-performance"   # リポジトリの中の E2 フォルダ（ルートが E2 なら "" にする）

IN_COLAB = "google.colab" in sys.modules


def is_e2_root(p):
    """cpu_basics.py と benchmark_utils.py があるフォルダを E2 のルートとみなす"""
    return (Path(p) / "cpu_basics.py").exists() and (Path(p) / "benchmark_utils.py").exists()


def find_e2_root():
    # 1) ローカル：notebooks/ の中、または E2 のルートで開いた場合
    for p in [Path.cwd(), Path.cwd().parent]:
        if is_e2_root(p):
            return p.resolve()
    if not IN_COLAB:
        return None
    # 2) Colab 方法A：clone
    if REPO_URL:
        dest = Path("/content/repo")
        if not dest.exists():
            subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(dest)], check=True)
        cand = dest / REPO_SUBDIR if REPO_SUBDIR else dest
        if is_e2_root(cand):
            return cand
    # 3) Colab 方法B：/content にある ZIP を展開（なければアップロードを求める）
    zips = glob.glob("/content/*.zip")
    if not zips:
        from google.colab import files
        print("E2 フォルダの ZIP を選んでアップロードしてください")
        zips = ["/content/" + name for name in files.upload()]
    for z in zips:
        with zipfile.ZipFile(z) as zf:
            zf.extractall("/content/e2_upload")
    for hit in glob.glob("/content/e2_upload/**/cpu_basics.py", recursive=True):
        if is_e2_root(Path(hit).parent):
            return Path(hit).parent
    return None


E2_ROOT = find_e2_root()
if E2_ROOT is None:
    raise RuntimeError("E2 のフォルダが見つかりません。上の説明（方法A／方法B）を確認してください。")
os.chdir(E2_ROOT)
if str(E2_ROOT) not in sys.path:
    sys.path.insert(0, str(E2_ROOT))
print("E2_ROOT:", E2_ROOT)
if not (E2_ROOT / "textbook" / "bsearch.py").exists():
    print("⚠️ textbook/ がありません。algorithm_benchmark.py の import が失敗します（上の注意を参照）。")

In [ ]:
import matplotlib.pyplot as plt

# グラフの共通スタイル（Colab には日本語フォントがないことが多いので、図の文字は英語にしている）
plt.rcParams.update({
    "figure.figsize": (7, 3.6), "axes.grid": True, "grid.alpha": 0.3,
    "axes.spines.top": False, "axes.spines.right": False, "lines.linewidth": 2,
})
# 色は「実装（ライブラリ）」ごとに固定する。デバイス（CPU / GPU / TPU）は線の種類と記号で区別する
COLORS = {"Python": "#eb6834", "NumPy": "#2a78d6", "PyTorch": "#1baf7a", "JAX": "#6250d6",
          "TensorFlow": "#eda100", "neutral": "#8a8985"}
DEVICE_STYLE = {"CPU": ("-", "o"), "GPU": ("--", "s"), "TPU": ("-.", "^")}

## 4. 使用するコード

`algorithm_benchmark.py` は、教科書のコード（`textbook/`）を**直接 import** しています。

| 関数 | 内容 |
| --- | --- |
| `linear_search` | 教科書の `seq_search` そのもの |
| `binary_search` | 教科書の `bin_search` を呼ぶ。**空の配列だけ先に -1 を返す**（教科書のコードは空の配列で `IndexError` になるため） |
| `SORTS` | 教科書の 3 つのソート（入力をコピーしてから並べ替える）、`sorted`、`np.sort` |
| `make_data` | `"random"`・`"sorted"`・`"reversed"`・`"few_unique"` の入力を作る |
| `search_experiment`・`sort_experiment`・`complexity_table` | 実験と、時間 ÷ $f(n)$ の計算 |

In [ ]:
import numpy as np
import algorithm_benchmark as ab
from benchmark_utils import format_table

SEED = 0
print("O(n^2) のソートの安全上限:", ab.QUADRATIC_MAX_N)

# 動作確認
data = [7, 3, 9, 1, 5, 3]
s = sorted(data)
print("linear_search(data, 9) =", ab.linear_search(data, 9))
print("binary_search(sorted, 9) =", ab.binary_search(s, 9), " sorted =", s)
print("binary_search([], 1) =", ab.binary_search([], 1))
for name, f in ab.SORTS.items():
    print(f"{name:15}", list(np.asarray(f(data)).tolist()))
assert all(list(np.asarray(f(data)).tolist()) == s for f in ab.SORTS.values())

## 5. 実行例

### 5-1. 線形探索と二分探索

見つかるキーと見つからないキーを半分ずつ探し、1 回あたりの時間を測ります。
`binary_with_sort_s` は「探索のたびにソートする」場合（データが毎回変わる状況）の時間です。

In [ ]:
SEARCH_SIZES = [100, 1_000, 10_000, 100_000]
search_rows = [ab.search_experiment(n, queries=100, repeats=3, seed=SEED) for n in SEARCH_SIZES]
print(format_table(search_rows, [("n", "n"), ("linear_search_s", "linear"), ("binary_search_s", "binary"),
                                 ("binary_with_sort_s", "sort + binary")]))

In [ ]:
plt.figure()
ns = np.array(SEARCH_SIZES)
for key, name, color in [("linear_search_s", "linear search O(n)", "#eb6834"),
                         ("binary_search_s", "binary search O(log n)", "#2a78d6"),
                         ("binary_with_sort_s", "sort + binary search", "#8a8985")]:
    plt.loglog(ns, [r[key] for r in search_rows], marker="o", ms=6, color=color, label=name)
# 理論曲線（最小の n の実測値に合わせて定数倍を決める）
lin0, bin0 = search_rows[0]["linear_search_s"], search_rows[0]["binary_search_s"]
plt.loglog(ns, lin0 * ns / ns[0], ":", color="#eb6834", lw=1)
plt.loglog(ns, bin0 * np.log2(ns) / np.log2(ns[0]), ":", color="#2a78d6", lw=1)
plt.xlabel("n"); plt.ylabel("time per search [s]"); plt.title("search: measured (solid) vs theory (dotted)")
plt.legend(fontsize=8); plt.show()

### 5-2. ソート：$O(n^2)$ と $O(n \log n)$

In [ ]:
SORT_SIZES = [100, 300, 1_000, 3_000]
sort_rows = []
for n in SORT_SIZES:
    sort_rows += ab.sort_experiment(n, "random", repeats=3, seed=SEED)
print(format_table(sort_rows, [("algorithm", "algorithm"), ("complexity", "O()"), ("n", "n"), ("time_s", "time")]))

In [ ]:
SORT_COLORS = {"bubble_sort": "#eb6834", "insertion_sort": "#eda100", "merge_sort": "#1baf7a",
               "sorted": "#2a78d6", "np.sort": "#6250d6"}
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 3.8))
for name in ab.SORTS:
    pts = [(r["n"], r["time_s"]) for r in sort_rows if r["algorithm"] == name and r["time_s"] is not None]
    xs, ys = zip(*pts)
    ax1.loglog(xs, ys, marker="o", ms=6, color=SORT_COLORS[name], label=f"{name} O({ab.COMPLEXITY[name]})")
ax1.set_xlabel("n"); ax1.set_ylabel("time [s]"); ax1.set_title("sorting time (random input)")
ax1.legend(fontsize=8)

# 時間 / f(n)：理論どおりなら水平に近くなる
for r_name in ab.SORTS:
    ct = [r for r in ab.complexity_table(sort_rows) if r["algorithm"] == r_name]
    ax2.semilogx([r["n"] for r in ct], [r["time_per_f"] / ct[0]["time_per_f"] for r in ct],
                 marker="o", ms=6, color=SORT_COLORS[r_name], label=r_name)
ax2.axhline(1, color="black", ls="--", lw=1)
ax2.set_xlabel("n"); ax2.set_ylabel("(time / f(n)), normalized to smallest n")
ax2.set_title("measured vs theoretical complexity"); ax2.legend(fontsize=8)
plt.tight_layout(); plt.show()

### 5-3. 入力の並びによる違い（最良・平均・最悪）

挿入ソートは、ソート済みの入力では内側のループがすぐ終わる（$O(n)$）が、逆順では最も遅い（$O(n^2)$）。バブルソート（`bubble_sort1`）は並びによらず比較の回数が同じです。

In [ ]:
ORDERS = ["sorted", "random", "reversed", "few_unique"]
n = 1_000
order_rows = []
for order in ORDERS:
    order_rows += ab.sort_experiment(n, order, names=["bubble_sort", "insertion_sort", "sorted"], repeats=3, seed=SEED)
print(format_table(order_rows, [("algorithm", "algorithm"), ("order", "order"), ("n", "n"), ("time_s", "time")]))

names = ["bubble_sort", "insertion_sort", "sorted"]
width = 0.25
plt.figure(figsize=(9, 3.8))
for k, name in enumerate(names):
    vals = [next(r["time_s"] for r in order_rows if r["algorithm"] == name and r["order"] == o) for o in ORDERS]
    plt.bar(np.arange(len(ORDERS)) + (k - 1) * width, vals, width, color=SORT_COLORS[name], label=name)
plt.xticks(range(len(ORDERS)), ORDERS); plt.yscale("log"); plt.ylabel("time [s] (log)")
plt.title(f"effect of input order (n = {n:,})"); plt.legend(); plt.show()

## 6. 結果の可視化

5-1〜5-3 のグラフを参照してください（データサイズと実行時間、線形探索と二分探索、$O(n^2)$ と $O(n \log n)$、理論計算量と実測）。

## 7. 結果の解釈（読み方のヒント）

- 両対数グラフの**傾き**がオーダーに対応します（$O(n)$ は 1、$O(n^2)$ は 2）。
- 「時間 / $f(n)$」が $n$ によらず一定に近ければ、理論計算量どおりに増えています。小さい $n$ では関数呼び出しなどの固定コストが目立ち、ずれることがあります。
- `sorted()` と `merge_sort` はどちらも $O(n \log n)$ ですが、**C 言語の実装と Python の実装**で定数倍が大きく違います。
- 二分探索は 1 回の探索は非常に速いが、**ソートの時間**を含めると線形探索より遅くなることがあります。何回探索するかで、どちらが得かが変わります。

## 8. 学生向け TODO

- [ ] **TODO 1**：データサイズ `SEARCH_SIZES`・`SORT_SIZES` を変更する（$O(n^2)$ のソートは `ab.QUADRATIC_MAX_N` まで）。
- [ ] **TODO 2**：入力データの並び `ORDERS` を変更する（例：「ほとんどソート済み」を自分で作る）。
- [ ] **TODO 3**：線形探索の最良（先頭にある）・最悪（存在しない）の場合を測る。
- [ ] **TODO 4**：ソート済み入力で、挿入ソートが $O(n)$ になることを、$n$ を変えて確かめる。
- [ ] **TODO 5**：「1 回ソートして $q$ 回探索する」場合、二分探索が線形探索より得になる $q$ を求める。
- [ ] **TODO 6**：理論計算量と実測時間のずれの原因を考察する。

## 9. 考察課題

1. $n$ を 10 倍にしたとき、線形探索・二分探索・バブルソート・`sorted()` の時間はそれぞれ何倍になりましたか。理論値と比べてください。
2. Python で書いたマージソート（$O(n \log n)$）と、C 言語の `sorted()`（$O(n \log n)$）の差は何倍でしたか。オーダーが同じなのに差が出る理由は何ですか。
3. 「GPU を使えばソートが速くなる」と言えますか。逐次的な依存関係（前の比較の結果で次が決まる）という観点から考えてください（Advanced C）。
4. アルゴリズムの改善と、実装の高速化（NumPy・C 言語・GPU）は、どのように違いますか。

## 10. 参考文献

- 柴田望洋『新・明解 Pythonで学ぶアルゴリズムとデータ構造（第2版）』SBクリエイティブ（第3章 探索、第6章 ソート）
- 参考 Notebook：`ai-programming-2026-team-project-1-starter/A_cpu_gpu_tpu/A_cpu_gpu_tpu_benchmark/starter.ipynb`（MIT License, Copyright (c) 2026 Daigo Takeuchi）
- NumPy ドキュメント <https://numpy.org/doc/>
- PyTorch ドキュメント：CUDA semantics（非同期実行と同期）<https://pytorch.org/docs/stable/notes/cuda.html>
- JAX ドキュメント：Asynchronous dispatch、Just-in-time compilation <https://docs.jax.dev/>

- E2 リポジトリの `README.md`